#Задача

Многокритериальная оптимизация:

1) Реализовать многокритериальную оптимизацию гиперпараметров нейронной сети, учитывая не только точность, но и время обучения.
2) Использовать метод весовых коэффициентов для объединения критериев.
3) Набор данных можете выбрать самостоятельно (MNIST).

**В данной работе будем минимзировать ошибку (1-accuracy) и время обучения.**

In [20]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, Input, Flatten
import random
import numpy as np
import time
from sklearn.metrics import accuracy_score

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train / 255.0
x_test = x_test / 255.0

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [37]:
# Функция приспособленности
def fitness_function(params):
    learning_rate, num_layers, num_neurons = params  # извлекаем гиперпараметры из входных параметров

    model = Sequential()  # создаем модель нейронной сети
    model.add(Input(shape=(28, 28))) # для MNIST
    model.add(Flatten()) # для MNIST
    #model.add(Input(shape=(x_train.shape[1],)))  # добавляем Input слой с указанием формы входных данных
    model.add(Dense(num_neurons, activation='relu'))  # добавляем первый слой с указанным количеством нейронов и ReLU активацией
    for _ in range(num_layers - 1):  # добавляем дополнительные скрытые слои
        model.add(Dense(num_neurons, activation='relu'))
    model.add(Dense(10, activation='softmax'))  # добавляем выходной слой с тремя нейронами и softmax активацией
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])  # компилируем модель с оптимизатором Adam и функцией потерь для многоклассовой классификации

    start_time = time.time() # засекаем время
    model.fit(x_train, y_train, epochs=5, verbose=0)  # обучаем модель на обучающих данных
    end_time = time.time() # конец обучения

    training_duration = end_time - start_time # время обучения
    y_pred_probs = model.predict(x_test, verbose=0)  # делаем предсказания на тестовых данных
    y_pred = np.argmax(y_pred_probs, axis=1)  # преобразуем вероятности в классы
    accuracy = accuracy_score(y_test, y_pred)  # вычисляем точность предсказаний
    error = 1 - accuracy # вычисляем ошибку (потому что минимизируем ошибку и время обучения)
    return (error, training_duration)  # возвращаем точность как значение приспособленности

In [38]:
def search_best_individual(w1, w2, fitness_scores):
    error = [i[0] for i in fitness_scores]
    time = [i[1] for i in fitness_scores]

    error_best = min(error)
    error_worst = max(error)
    time_best = min(time)
    time_worst = max(time)

    error_norm = []
    time_norm = []

    for i in range(len(error)):
        # Защита от деления на ноль
        err_norm = (error[i] - error_best) / (error_worst - error_best) if error_worst != error_best else 0
        time_norm_val = (time[i] - time_best) / (time_worst - time_best) if time_worst != time_best else 0
        error_norm.append(err_norm)
        time_norm.append(time_norm_val)

    res = [w1*error_norm[i] + w2*time_norm[i] for i in range(len(error_norm))]
    best_index = res.index(min(res))

    return best_index, fitness_scores[best_index]

In [39]:
def compare_individuals(w1, w2, ind1, ind2):
    error_best = min([ind1[0], ind2[0]])
    error_worst = max([ind1[0], ind2[0]])
    time_best = min([ind1[1], ind2[1]])
    time_worst = max([ind1[1], ind2[1]])

    # Защита от деления на ноль
    err_range = error_worst - error_best
    time_range = time_worst - time_best

    err_1 = (ind1[0] - error_best) / err_range if err_range != 0 else 0
    time_1 = (ind1[1] - time_best) / time_range if time_range != 0 else 0
    err_2 = (ind2[0] - error_best) / err_range if err_range != 0 else 0
    time_2 = (ind2[1] - time_best) / time_range if time_range != 0 else 0

    crit_1 = w1*err_1 + w2*time_1
    crit_2 = w1*err_2 + w2*time_2

    return crit_1 < crit_2

In [40]:
# Генетический алгоритм
def genetic_algorithm(population_size, num_generations, mutation_rate, w1, w2):
    population = []  # инициализируем популяцию
    for _ in range(population_size):  # создаем начальную популяцию
        learning_rate = random.uniform(0.001, 0.1)  # случайно выбираем learning rate
        num_layers = random.randint(1, 5)  # случайно выбираем количество слоев
        num_neurons = random.randint(50, 200)  # случайно выбираем количество нейронов
        population.append((learning_rate, num_layers, num_neurons))  # добавляем особь в популяцию

    best_fitness = (1.0, 200.0)
    best_individual = None

    for generation in range(num_generations):  # проходим через заданное количество поколений
        print(f"\nПоколение {generation + 1}/{num_generations}")

        # Вычисляем приспособленность каждой особи
        fitness_scores = []
        for idx, individual in enumerate(population):
            fitness = fitness_function(individual)
            fitness_scores.append(fitness)
            print(f"Особь {idx + 1}: LR={individual[0]:.4f}, "
                  f"Слои={individual[1]}, Нейроны={individual[2]}, "
                  f"Ошибка={fitness[0]:.4f} " , f"Время={fitness[1]:.4f}")

        # Находим лучшую особь в текущем поколении
        '''current_best_idx = np.argmax(fitness_scores)
        current_best_fitness = fitness_scores[current_best_idx]
        current_best_individual = population[current_best_idx]'''

        current_best_idx, current_best_fitness = search_best_individual(w1, w2, fitness_scores)
        current_best_individual = population[current_best_idx]

        # Обновляем лучшую особь за все время
        '''if current_best_fitness > best_fitness:
            best_fitness = current_best_fitness
            best_individual = current_best_individual
            print(f"\nНовый лучший результат!")
            print(f"Параметры: LR={best_individual[0]:.4f}, "
                  f"Слои={best_individual[1]}, Нейроны={best_individual[2]}")
            print(f"Точность: {best_fitness:.4f}")'''

        if compare_individuals(w1, w2, current_best_fitness, best_fitness):
            best_fitness = current_best_fitness
            best_individual = current_best_individual
            print(f"\nНовый лучший результат!")
            print(f"Параметры: LR={best_individual[0]:.4f}, "
                  f"Слои={best_individual[1]}, Нейроны={best_individual[2]}")
            print(f"Ошибка: {best_fitness[0]:.4f} ", f"Время: {best_fitness[1]:.4f}")

        selected_population = []  # инициализируем выборку для следующего поколения
        for _ in range(population_size):  # выбираем особей для следующего поколения
            '''tournament = random.sample(list(zip(population, fitness_scores)), 3)  # проводим турнир из 3 случайных особей
            winner = max(tournament, key=lambda x: x[1])[0]  # выбираем победителя турнира
            selected_population.append(winner)  # добавляем победителя в выборку'''
            tournament = random.sample(list(zip(population, fitness_scores)), 3)  # проводим турнир из 3 случайных особей
            f_scores = []
            for i in tournament:
                  f_scores.append(i[1])
            current_best_tourn_idx = search_best_individual(w1, w2, f_scores)[0]
            winner = tournament[current_best_tourn_idx][0]
            selected_population.append(winner)


        new_population = []  # инициализируем новую популяцию
        for i in range(0, population_size, 2):  # проходим по парам особей
            parent1, parent2 = selected_population[i], selected_population[i + 1]  # выбираем родителей
            child1, child2 = crossover(parent1, parent2)  # применяем кроссовер для создания потомков
            child1 = mutate(child1, mutation_rate)  # применяем мутацию к первому потомку
            child2 = mutate(child2, mutation_rate)  # применяем мутацию ко второму потомку
            new_population.append(child1)  # добавляем первого потомка в новую популяцию
            new_population.append(child2)  # добавляем второго потомка в новую популяцию

        population = new_population  # обновляем популяцию

    return best_individual, best_fitness  # возвращаем лучшую особь лучшую точность

def crossover(parent1, parent2):
    child1 = (parent1[0], parent2[1], parent1[2])  # создаем первого потомка путем обмена параметрами между родителями
    child2 = (parent2[0], parent1[1], parent2[2])  # создаем второго потомка путем обмена параметрами между родителями
    return child1, child2  # возвращаем потомков

def mutate(individual, mutation_rate):
    if random.random() < mutation_rate:
        # Используем те же диапазоны, что и при инициализации
        return (random.uniform(0.001, 0.1),
                random.randint(1, 5),
                random.randint(50, 200))
    return individual

In [41]:
# Запуск генетического алгоритма
population_size = 10  # размер популяции
num_generations = 10  # количество поколений
mutation_rate = 0.1  # вероятность мутации
w1 = 0.7 # вес точности (для метода весовых коэффициентов)
w2 = 0.3 # вес времени (для метода весовых коэффициентов)

best_params, best_criteria = genetic_algorithm(population_size, num_generations, mutation_rate, w1, w2)
print("\nИтоговые результаты:")
print(f"Лучшие гиперпараметры: LR={best_params[0]:.4f}, "
      f"Слои={best_params[1]}, Нейроны={best_params[2]}")
print(f"Лучшая (минимальная) ошибка: {best_criteria[0]:.4f}")
print(f"Лучшая точность: {1 - best_criteria[0]:.4f}")
print(f"Лучшее время: {best_criteria[1]:.4f}")


Поколение 1/10
Особь 1: LR=0.0241, Слои=2, Нейроны=93, Ошибка=0.0683  Время=23.9848
Особь 2: LR=0.0341, Слои=5, Нейроны=141, Ошибка=0.6043  Время=31.2074
Особь 3: LR=0.0010, Слои=4, Нейроны=105, Ошибка=0.0283  Время=25.7684
Особь 4: LR=0.0992, Слои=4, Нейроны=68, Ошибка=0.8865  Время=25.9137
Особь 5: LR=0.0707, Слои=2, Нейроны=167, Ошибка=0.7171  Время=23.6458
Особь 6: LR=0.0151, Слои=3, Нейроны=169, Ошибка=0.0570  Время=24.7774
Особь 7: LR=0.0474, Слои=1, Нейроны=73, Ошибка=0.1214  Время=24.2495
Особь 8: LR=0.0224, Слои=4, Нейроны=158, Ошибка=0.1385  Время=25.5686
Особь 9: LR=0.0746, Слои=5, Нейроны=91, Ошибка=0.8968  Время=26.4736
Особь 10: LR=0.0769, Слои=2, Нейроны=85, Ошибка=0.8184  Время=23.8470

Новый лучший результат!
Параметры: LR=0.0241, Слои=2, Нейроны=93
Ошибка: 0.0683  Время: 23.9848

Поколение 2/10
Особь 1: LR=0.0151, Слои=2, Нейроны=169, Ошибка=0.0495  Время=23.4044
Особь 2: LR=0.0707, Слои=3, Нейроны=167, Ошибка=0.5556  Время=24.1174
Особь 3: LR=0.0010, Слои=4, Нейроны